In [ ]:
import tensorflow as tf
import numpy as np
import h5py

print("TF:", tf.__version__)
print("NumPy:", np.__version__)
print("h5py:", h5py.__version__)


: 

In [ ]:
# imports
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import random

: 

In [ ]:
import tensorflow as tf

print("TF Version:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())

In [ ]:
class_df = pd.read_csv("E:/Datasets/AIA-image-classification/images.csv")
folder = "E:/Datasets/AIA-image-classification/images"
class_df.head(5)

In [ ]:
clean_names = []
clean_paths = []
clean_labels = []
missing_files = []

for index, row in class_df.iterrows():
    image_id = str(row['image_name'])
    label = row['label']
    
    full_path = os.path.join(folder, image_id)
    
    if os.path.exists(full_path):
        clean_names.append(image_id)
        clean_paths.append(full_path)
        clean_labels.append(label)
    else:
        missing_files.append(image_id)

# ✅ Create dataframe (ALL columns aligned)
df = pd.DataFrame({
    'image_name': clean_names,
    'image_paths': clean_paths,
    'label': clean_labels
})

# Required for generator
df['label'] = df['label'].astype(str)

print("Missing files:", missing_files)
print(df.info())
print(df['label'].value_counts())
df.head(10)

In [ ]:
from PIL import Image

bad_images = []
for image in df['image_paths']:
    try:
        img = Image.open(image)  
        img.verify()             
    except:
        bad_images.append(image)

len(bad_images)

In [ ]:
train_iterator = train_generator.flow_from_dataframe(
    train_df,
    x_col='image_paths',
    y_col='label',
    target_size=(128, 128),   # 🔥 smaller = faster + easier
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_iterator = val_generator.flow_from_dataframe(
    val_df,
    x_col='image_paths',
    y_col='label',
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    
    # Conv Block 1
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),
    
    # Conv Block 2
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    
    # Conv Block 3
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    # Conv Block 3
    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),


    layers.GlobalAveragePooling2D(),
    
    # Output
    layers.Dense(6, activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_iterator,
    validation_data=val_iterator,
    epochs=5
)

In [ ]:
# input split
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_generator = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_generator = ImageDataGenerator(
    rescale=1./255
)

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model
base_model.trainable = False

In [ ]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)

output = layers.Dense(6, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping


early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model.keras",
    save_best_only=True
)

In [ ]:
history = model.fit(
    train_iterator,
    validation_data=val_iterator,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
loss, acc = model.evaluate(val_iterator)

print("🔥 Validation Accuracy:", acc)

In [ ]:
train_iterator = train_generator.flow_from_dataframe(
    train_df,
    x_col='image_paths',
    y_col='label',
    target_size=(224, 224),   # EfficientNet default
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_iterator = val_generator.flow_from_dataframe(
    val_df,
    x_col='image_paths',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.metrics import Precision, Recall, Accuracy

def create_model():
    model = Sequential([
        Conv2D(16, (3, 3), activation='relu', input_shape=(256, 256, 3)),
        MaxPooling2D(),
        BatchNormalization(),
        
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D(),
        BatchNormalization(),
    
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(),
        BatchNormalization(),
    
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(),
        BatchNormalization(),
    
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        layers.Dropout(0.5),
        Dense(6, activation='softmax')    
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy', Precision(), Recall()])
    return model

model = create_model()
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping


early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model.keras",
    save_best_only=True
)

In [ ]:
classes = sorted(df['label'].unique())
print("Classes:", classes)

In [ ]:
from sklearn.model_selection import KFold

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

acc_scores = []
precision_scores = []
recall_scores = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(df)):
    
    print("Train:", train_iterator.class_indices)
    print("Val:", val_iterator.class_indices)
    print(f"\n🔥 Fold {fold+1}")

    # Split dataframe
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    # Generators
    train_iterator = train_generator.flow_from_dataframe(
    train_df,
    x_col='image_paths',
    y_col='label',
    target_size=(256, 256),
    batch_size=256,
    class_mode='categorical',
    classes=classes
    )

    val_iterator = val_generator.flow_from_dataframe(
        val_df,
        x_col='image_paths',
        y_col='label',
        target_size=(256, 256),
        batch_size=256,
        class_mode='categorical',
        classes=classes
    )
    
    model = create_model()

    # Train
    model.fit(
        train_iterator,
        validation_data=val_iterator,
        epochs=20,
        callbacks=[early_stop])

    # Evaluate
    loss, acc, precision, recall = model.evaluate(val_iterator)
    print("Accuracy:", acc)
    # Store scores
    acc_scores.append(acc)
    precision_scores.append(precision)
    recall_scores.append(recall)
print('🔥 Done !')

In [ ]:
# Final result
print("🔥 Average Accuracy:", sum(acc_scores)/len(acc_scores))
print("🔥 Precision:", precision_scores)
print("🔥 Recall:", recall_scores)